# Domain-Specific Fine-Tuning

A practical refresher on **domain-specific fine-tuning** — taking a general-purpose
foundation model (Llama, Mistral, Qwen, GPT-class) and adapting it to a *narrow domain*
(medicine, law, finance, customer support, a single codebase) so it speaks the domain's
vocabulary, follows its conventions, and stays reliable on its tasks.

Fine-tuning sits **after** pre-training in the LLM lifecycle. Where *continued
pre-training* injects raw knowledge with a self-supervised objective on unlabeled text,
**fine-tuning** is **supervised**: it learns from curated `(prompt, completion)` pairs —
instruction/response, chat turns, or classification labels — to shape *behavior* and
*format*. This notebook focuses on the MLOps reality of doing that well: data curation,
LoRA/QLoRA vs. full fine-tuning, training infrastructure, evaluation, serving, and the
pitfalls (catastrophic forgetting, overfitting, data leakage) that bite in production.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction <a id="introduction"></a>

Domain-specific fine-tuning is the supervised stage of model adaptation. You start from
a capable base (or instruct) model and continue training it on a **labeled dataset that
represents your domain's tasks**, so the model's outputs match the style, terminology,
and decision boundaries your users expect.

### What is it?

Concretely, you minimize the next-token cross-entropy loss on curated examples — most
often chat-formatted `{"messages": [...]}` or `{"prompt": ..., "completion": ...}`
records. The loss is typically computed **only on the completion (assistant) tokens**, so
the model learns *what to answer*, not *how to repeat the question*. Modern practice rarely
updates all weights; instead **parameter-efficient fine-tuning (PEFT)** — chiefly
**LoRA** and its quantized variant **QLoRA** — trains a small set of adapter matrices
(<1% of parameters) and leaves the base frozen.

### Why use it?

- **Behavioral alignment** — the model reliably produces your required format (JSON
  schema, SOAP notes, ICD codes, your support tone) instead of generic prose.
- **Quality at lower cost** — a fine-tuned small model (7–8B) often matches a much larger
  prompted model on a *narrow* task, cutting inference cost and latency.
- **Latency & determinism** — no need for huge few-shot prompts; the behavior is baked in,
  shrinking the prompt and the bill.
- **Data moat & privacy** — proprietary labeled data becomes a capability you own and can
  run in your own VPC, with no examples leaking into a vendor prompt.

### When to use it?

- You have (or can build) **hundreds-to-thousands of high-quality labeled examples** for a
  stable, well-defined task.
- Prompt engineering and RAG have plateaued — the failures are about *behavior/format/tone*,
  not missing facts.
- You need **consistent structured output**, lower latency, or on-prem deployment.

**When *not* to**: when the gap is *missing knowledge* that changes often → prefer
**RAG**; when the gap is *the base model never saw your domain's language at all* (rare
jargon, a new programming language) → consider **continued pre-training first**, then
fine-tune. Fine-tuning teaches *behavior*, not fresh facts.

## Key Features <a id="key-features"></a>

### What distinguishes domain-specific fine-tuning

| Capability | What it means | Why it matters |
|------------|---------------|----------------|
| **Supervised, label-on-completion** | Loss is masked to assistant tokens of curated `(prompt, completion)` pairs | Teaches the *response behavior* directly, not just the input distribution |
| **PEFT / LoRA adapters** | Train low-rank `A·B` matrices on top of frozen weights (~0.1–1% of params) | A single 24 GB GPU can fine-tune a 7B model; adapters are tens of MB, easy to version and swap |
| **QLoRA (4-bit base + LoRA)** | Load the frozen base in NF4 4-bit, train adapters in bf16 | Cuts memory ~4×, so a 70B model fits on a single 48–80 GB GPU |
| **Multi-adapter serving** | Hot-swap many task adapters over one base in vLLM/TGI | One GPU pool serves dozens of tenants/tasks without re-loading the base |
| **Format & tone control** | Reliable JSON/schema, domain register, refusal policy | Replaces brittle prompt scaffolding with learned behavior |
| **Reproducible artifacts** | Deterministic config + seed + data hash → registered adapter | Auditable, rollback-able model lineage for regulated domains |

## Architecture Overview <a id="architecture"></a>

Fine-tuning is the same transformer and the same causal-LM loss as pre-training, but on
**curated labeled data**, with the loss **masked to the response**, and (for PEFT) with
the base weights **frozen** while small adapters learn.

```
                ┌──────────────────────────────────────────────────────┐
  Raw sources → │  Data curation:                                       │
  (tickets,     │   collect → clean/PII-scrub → dedup → quality filter →│
   docs, SME    │   format to chat/JSONL → train/val/test split →       │
   labels)      │   decontaminate vs. eval set                          │
                └───────────────────────────┬──────────────────────────┘
                                            │  prompt/completion JSONL
                                            ▼
   ┌────────────────────┐      ┌──────────────────────────────────────┐
   │ Frozen base model  │─────▶│  Trainer (SFTTrainer / Axolotl /      │
   │ (4-bit for QLoRA)  │      │  torchtune): mask loss → assistant    │
   └────────────────────┘      │  tokens; update LoRA adapters only    │
            ▲                   └───────────────────┬──────────────────┘
            │ low-rank ΔW = B·A                     │ checkpoints + metrics
            └───────────────────────────────────────┘
                                            │
                ┌───────────────────────────▼──────────────────────────┐
                │  Eval: held-out test + domain rubric + LLM-judge +    │
                │  regression vs. base (catastrophic-forgetting check)  │
                └───────────────────────────┬──────────────────────────┘
                                            │ pass gate
                                            ▼
                ┌──────────────────────────────────────────────────────┐
                │  Register adapter → (optional merge) → serve via      │
                │  vLLM/TGI with adapter hot-swap → monitor in prod     │
                └──────────────────────────────────────────────────────┘
```

### Components

1. **Data pipeline**: the dominant lever. Curation, deduplication, PII scrubbing,
   chat formatting, and **decontamination** (removing eval examples from training).
2. **Base model**: an instruct or base checkpoint; quantized to 4-bit (NF4) for QLoRA.
3. **PEFT adapters (LoRA)**: low-rank matrices injected into attention/MLP projections;
   the only weights that receive gradients.
4. **Trainer**: TRL `SFTTrainer`, Axolotl, or torchtune — handles loss masking, packing,
   gradient checkpointing, and distributed training (FSDP/DeepSpeed).
5. **Evaluation harness**: held-out domain test set + general-capability regression suite.
6. **Serving layer**: vLLM/TGI with LoRA hot-swapping, or a merged full-weight model.

## Installation <a id="installation"></a>

### Prerequisites

- **A CUDA GPU.** QLoRA of a 7B model fits in ~10–12 GB; a 13B in ~16 GB; a 70B in
  ~48 GB. Full fine-tuning needs roughly **16–20 GB per billion parameters** (weights +
  gradients + Adam states), so prefer LoRA/QLoRA unless you have a multi-GPU FSDP setup.
- **CUDA 12.x**, recent NVIDIA driver, PyTorch 2.x.
- Core libraries: `transformers`, `peft`, `trl`, `datasets`, `accelerate`, plus
  `bitsandbytes` for 4-bit QLoRA. `flash-attn` is optional but speeds up training.

### Installation Steps

**Note**: Uncomment the following cell to install (e.g. in Google Colab or a fresh venv).

In [ ]:
# Uncomment to install the fine-tuning stack.
# !pip install -U "transformers>=4.44" "peft>=0.12" "trl>=0.9" \
#     "datasets>=2.20" "accelerate>=0.33" "bitsandbytes>=0.43"
# Optional (Ampere/Hopper GPUs, faster attention):
# !pip install flash-attn --no-build-isolation

## Basic Usage <a id="basic-usage"></a>

### Quick Start Example

A minimal QLoRA fine-tune with TRL's `SFTTrainer`. The pattern is: load a 4-bit base,
attach a LoRA config, point the trainer at a chat-formatted dataset, and train. Loss is
automatically masked to the assistant turns when you use a chat template.

In [ ]:
"""End-to-end QLoRA fine-tune skeleton (runs on a single ~16 GB GPU for a 7B model).

This cell is illustrative — running it requires a GPU and downloads a multi-GB base
model, so it is left un-executed in the notebook.
"""
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
import torch

BASE = "mistralai/Mistral-7B-Instruct-v0.3"

# 1) Domain dataset in chat format: each row is {"messages": [{role, content}, ...]}.
#    Swap this for your own JSONL: load_dataset("json", data_files="train.jsonl").
ds = load_dataset("json", data_files={"train": "domain_train.jsonl",
                                      "test": "domain_val.jsonl"})

# 2) Load base in 4-bit (QLoRA) to slash memory.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)

# 3) LoRA: train only low-rank adapters on attention + MLP projections.
peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# 4) Train. SFTTrainer applies the chat template and masks loss to assistant tokens.
cfg = SFTConfig(
    output_dir="out/domain-mistral-7b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch = 16
    learning_rate=2e-4,                 # LoRA tolerates a higher LR than full FT
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="steps", eval_steps=100, save_steps=100,
    packing=True,                       # pack short samples -> better GPU utilization
    max_seq_length=2048,
)
trainer = SFTTrainer(model=model, args=cfg, peft_config=peft_cfg,
                     train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tok)
trainer.train()
trainer.save_model("out/domain-mistral-7b")   # saves only the LoRA adapter (~tens of MB)
print("Done - adapter saved.")

## Advanced Features <a id="advanced-features"></a>

### Going beyond a basic LoRA run

#### Feature 1: Inference with — and without — the adapter

LoRA adapters are loaded *on top of* the base at inference. For latency-critical serving
you can **merge** the adapter into the base weights, producing a standalone model with no
runtime overhead (at the cost of losing easy hot-swapping).

In [ ]:
"""Load the adapter for inference, then optionally merge it into the base."""
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER = "out/domain-mistral-7b"

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)        # base + adapter

msgs = [{"role": "user", "content": "Summarize this discharge note as a SOAP entry: ..."}]
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
out = model.generate(inputs.to(model.device), max_new_tokens=256, temperature=0.2)
print(tok.decode(out[0], skip_special_tokens=True))

# Merge for zero-overhead serving (no PEFT layers at runtime):
merged = model.merge_and_unload()
merged.save_pretrained("out/domain-mistral-7b-merged")
tok.save_pretrained("out/domain-mistral-7b-merged")

#### Feature 2: Knobs that actually move the needle

- **Rank `r` and `lora_alpha`**: higher rank (32–64) = more capacity for harder domain
  shifts; keep `alpha ≈ 2·r` as a starting heuristic. Start small — rank 8–16 is enough
  for most format/tone tasks.
- **`target_modules`**: adapting *all* linear layers (attention **and** MLP) generally
  beats attention-only for knowledge-heavy domains, at a modest memory cost.
- **NEFTune** (noisy embedding fine-tuning): add uniform noise to input embeddings during
  training (`neftune_noise_alpha=5`) — a cheap, well-documented boost to instruction
  quality.
- **DoRA / rsLoRA**: drop-in LoRA variants (`use_dora=True`, `use_rslora=True`) that often
  improve stability and quality at high rank for little extra cost.
- **Preference tuning after SFT**: once behavior is right, **DPO/ORPO** on
  `(chosen, rejected)` pairs sharpens it toward human preference without a reward model.

## Use Cases <a id="use-cases"></a>

### Real-world applications

#### Use Case 1: Clinical note structuring (healthcare)

- **Context**: Clinicians dictate free-text encounters; downstream billing/EHR systems
  need structured SOAP notes with ICD-10 codes.
- **Implementation**: ~3k SME-reviewed `(transcript -> structured note)` pairs; QLoRA on an
  8B instruct model, loss masked to the structured output; PII scrubbed and the model
  served entirely in-VPC for HIPAA.
- **Results**: Reliable schema-valid output and correct domain register that a prompted
  base model produced only ~70% of the time — at a fraction of a frontier-model API cost.

#### Use Case 2: Support-ticket triage & drafting (SaaS)

- **Context**: Replace a brittle 4k-token few-shot prompt that classified tickets and
  drafted replies in the company voice.
- **Implementation**: 10k historical `(ticket -> {category, priority, draft_reply})`
  examples; LoRA fine-tune; one base model with a per-product **adapter hot-swapped** in
  vLLM.
- **Results**: Prompt shrank from 4k to ~300 tokens (lower latency and cost), consistent
  tone, and new product lines added by training another small adapter — no base reload.

#### Use Case 3: Code assistant for a proprietary framework

- **Context**: A general code model doesn't know an internal framework's APIs/idioms.
- **Implementation**: Continued pre-training on the codebase to inject knowledge, **then**
  fine-tune on `(task -> code)` pairs and PR review comments to shape behavior.
- **Results**: Idiomatic completions using the right internal helpers, fewer hallucinated
  APIs, and review comments that match team conventions.

## Best Practices <a id="best-practices"></a>

1. **Data quality beats data quantity.** A few hundred clean, diverse, correctly-labeled
   examples outperform tens of thousands of noisy ones. Have SMEs review samples; remove
   contradictions and near-duplicates. This is where the wins live.
2. **Always hold out a real test set — and decontaminate.** Split *before* augmentation,
   and ensure no test example (or near-duplicate) leaks into training. Leakage is the
   #1 cause of "great metrics, bad production."
3. **Match the chat template and mask the loss to the response.** Use the base model's
   exact template; computing loss over prompt tokens teaches the model to parrot inputs.
4. **Start with LoRA/QLoRA, escalate only if needed.** Reach for full fine-tuning or
   continued pre-training only when adapters demonstrably plateau on a knowledge-heavy gap.
5. **Tune conservatively.** 1–3 epochs, LR ~1e-4–2e-4 for LoRA, cosine schedule with
   warmup. Watch eval loss — if it turns up while train loss keeps falling, you're
   overfitting; stop early.
6. **Test for catastrophic forgetting.** Run a general-capability suite (e.g. a slice of
   MMLU/IFEval) before and after — domain gains shouldn't wreck general ability. Mixing
   in ~5–10% general instruction data helps.
7. **Version everything as one artifact.** Pin base model revision, data hash, config,
   seed, and library versions so any adapter is reproducible and auditable.

## Common Pitfalls <a id="pitfalls"></a>

1. **Expecting fine-tuning to add facts.** SFT shapes *behavior and format*, not fresh,
   frequently-changing knowledge. *Avoid*: use **RAG** for knowledge; fine-tune for how to
   *use* it. If the base genuinely lacks the domain language, do continued pre-training.
2. **Data leakage / contamination.** Test examples present in training inflate metrics and
   collapse in prod. *Avoid*: split before any augmentation; run exact + near-dup
   (e.g. MinHash) decontamination between train and eval.
3. **Catastrophic forgetting.** Over-narrow training erodes general reasoning, safety, and
   formatting. *Avoid*: low LR, few epochs, mix in general data, and run a regression suite.
4. **Overfitting on a small set.** Too many epochs makes the model memorize and parrot.
   *Avoid*: early stopping on eval loss, dropout, and held-out monitoring.
5. **Wrong/missing chat template or unmasked loss.** Produces broken turn structure and a
   model that echoes prompts. *Avoid*: use the tokenizer's chat template; let the trainer
   mask completion-only loss.
6. **Ignoring class/format imbalance.** A skewed label distribution biases the model.
   *Avoid*: balance or weight the dataset; report per-class metrics, not just accuracy.

## Performance Optimization <a id="performance"></a>

### Optimizing training and the resulting model

#### Configuration Tuning

Key knobs, and how to set them:

- **Quantization (QLoRA NF4)**: ~4× memory reduction; the standard way to fit large bases
  on one GPU. Use `bnb_4bit_use_double_quant=True` and bf16 compute dtype.
- **Gradient checkpointing**: trades ~20–30% compute for a large activation-memory saving —
  almost always worth it for fine-tuning.
- **Sequence packing**: concatenate short examples to fill `max_seq_length`, dramatically
  improving GPU utilization on datasets with short samples.
- **Effective batch size**: `per_device_batch_size × grad_accum × num_gpus`. Raise grad
  accumulation rather than per-device batch when VRAM is tight.
- **Flash Attention 2 + bf16**: faster, lower-memory attention on Ampere/Hopper GPUs.
- **Multi-GPU**: FSDP or DeepSpeed ZeRO-3 for full fine-tuning; LoRA usually scales fine
  with plain DDP/accelerate.

The cell below estimates the rough memory footprint to pick a strategy before you burn
GPU hours.

In [ ]:
"""Rough VRAM estimator: full fine-tune vs. LoRA vs. QLoRA for a given model size."""

def vram_estimate_gb(params_billions: float) -> dict:
    p = params_billions
    return {
        "inference_bf16":   round(2.0 * p, 1),            # 2 bytes/param
        "inference_4bit":   round(0.5 * p, 1),            # ~0.5 bytes/param (NF4)
        # Full FT: weights(2) + grads(2) + Adam m,v(8) ~= 12-16 B/param + activations.
        "full_finetune":    round(16.0 * p, 1),
        # LoRA: frozen bf16 base + tiny adapter states + activations.
        "lora_bf16_base":   round(2.0 * p + 0.15 * p + 2.0, 1),
        # QLoRA: 4-bit frozen base + adapter + activations.
        "qlora_4bit_base":  round(0.5 * p + 0.15 * p + 2.0, 1),
    }


for size in (7, 13, 70):
    est = vram_estimate_gb(size)
    print(f"{size}B model (approx GB VRAM):")
    for k, v in est.items():
        print(f"  {k:<18} {v:>6} GB")
    print()

## Production Deployment <a id="deployment"></a>

### Deploying a fine-tuned model

Two serving strategies:

1. **Merged model** — bake the adapter into the base (`merge_and_unload`) and serve like
   any model. Zero adapter overhead, simplest to operate; one model per task.
2. **Adapter hot-swap** — keep one base in GPU memory and load many LoRA adapters on top
   (vLLM `--enable-lora`, TGI `LORA_ADAPTERS`). One GPU pool serves many tasks/tenants.

#### Serve with vLLM (multi-adapter)

```bash
# One base, multiple hot-swappable LoRA adapters; pick the adapter via the `model` field.
vllm serve mistralai/Mistral-7B-Instruct-v0.3 \
    --enable-lora \
    --lora-modules support=/models/adapters/support \
                   clinical=/models/adapters/clinical \
    --max-lora-rank 32 --dtype bfloat16
```

#### Docker Deployment

```dockerfile
FROM vllm/vllm-openai:latest
# Bake adapters into the image (or mount them at runtime).
COPY adapters/ /models/adapters/
ENV HF_HOME=/models/hf
EXPOSE 8000
ENTRYPOINT ["python", "-m", "vllm.entrypoints.openai.api_server", \
            "--model", "mistralai/Mistral-7B-Instruct-v0.3", \
            "--enable-lora", \
            "--lora-modules", "support=/models/adapters/support"]
```

#### Kubernetes Deployment

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: domain-llm
spec:
  replicas: 2
  selector:
    matchLabels: { app: domain-llm }
  template:
    metadata:
      labels: { app: domain-llm }
    spec:
      containers:
        - name: vllm
          image: registry.internal/domain-llm:1.4.0   # immutable, versioned tag
          args: ["--model", "mistralai/Mistral-7B-Instruct-v0.3",
                 "--enable-lora",
                 "--lora-modules", "support=/models/adapters/support"]
          ports: [{ containerPort: 8000 }]
          resources:
            limits:
              nvidia.com/gpu: 1
          readinessProbe:
            httpGet: { path: /health, port: 8000 }
            initialDelaySeconds: 60
---
apiVersion: v1
kind: Service
metadata:
  name: domain-llm
spec:
  selector: { app: domain-llm }
  ports: [{ port: 80, targetPort: 8000 }]
```

## Monitoring and Observability <a id="monitoring"></a>

### Monitoring a fine-tuned model in production

A fine-tuned model degrades silently when the live input distribution drifts away from the
training data. Monitor both **systems** and **quality** signals.

#### Key Metrics to Track

- **Quality / drift**: schema-validity rate (for structured output), domain rubric pass
  rate from a sampled **LLM-judge** or human review, refusal/empty-output rate, and
  embedding-based **input drift** vs. the training distribution.
- **Behavioral guardrails**: hallucination/format-error rate, off-policy responses, and a
  small **golden eval set** replayed on every deploy (canary) to catch regressions.
- **Systems**: tokens/sec throughput, p50/p95/p99 latency, GPU utilization & memory,
  per-adapter request mix, and cost per 1k tokens.
- **Feedback loop**: thumbs-up/down and corrections funneled back as candidate training
  data for the next iteration.

#### Logging Best Practices

- Log **structured records** (request id, adapter name, model+adapter version, token
  counts, latency, judge score) — never raw PII; redact at the edge.
- Use appropriate levels and sample heavy payloads; keep prompts/outputs in a governed,
  access-controlled store for audit and for building the next dataset.
- Tag every log line with the **artifact version** (base revision + adapter hash) so
  quality changes are attributable to a specific model.

## Troubleshooting <a id="troubleshooting"></a>

### Common issues and fixes

#### Issue 1: CUDA out of memory during training

**Symptoms**: `torch.cuda.OutOfMemoryError` early in training or at the first eval step.

**Cause**: Batch/sequence length too large for the GPU, or full-precision base when you
intended QLoRA.

**Solution**: Enable 4-bit (`load_in_4bit=True`), turn on `gradient_checkpointing`, lower
`per_device_train_batch_size` and raise `gradient_accumulation_steps`, reduce
`max_seq_length`, and ensure eval batch size is small.

#### Issue 2: Eval loss rises while train loss keeps falling

**Symptoms**: Training loss drops smoothly but validation loss bottoms out then climbs;
outputs start parroting training phrasing.

**Cause**: Overfitting — too many epochs, LR too high, or dataset too small/repetitive.

**Solution**: Reduce epochs (try 1–2), lower LR, add `lora_dropout`, enable early stopping
on eval loss, and diversify/expand the dataset.

#### Issue 3: Model improves on the domain but breaks general tasks

**Symptoms**: Great domain metrics, but it now fails basic reasoning, formatting, or
safety on out-of-domain prompts.

**Cause**: Catastrophic forgetting from over-narrow, over-long training.

**Solution**: Lower LR and epochs, mix in ~5–10% general instruction data, use a lower
LoRA rank, and gate releases on a general-capability regression suite.

#### Issue 4: Garbled outputs or the model echoes the prompt

**Symptoms**: Broken turn structure, special tokens leaking, or the assistant repeating
the user's message.

**Cause**: Wrong/missing chat template, or loss not masked to completion tokens.

**Solution**: Apply the tokenizer's exact `chat_template`, set the pad token, and let the
trainer mask loss to assistant tokens (use a chat-format dataset with `SFTTrainer`).

## Comparison with Alternatives <a id="comparison"></a>

### How domain-specific fine-tuning compares

| Dimension | Domain Fine-Tuning (SFT/LoRA) | Prompt Eng. + Few-shot | RAG | Continued Pre-Training |
|-----------|-------------------------------|------------------------|-----|------------------------|
| Adds **behavior/format/tone** | Strong | Limited, brittle | No | No |
| Adds **fresh/changing facts** | No | Via context | Strong | Static snapshot |
| Up-front cost | Medium (data + GPU) | Low | Medium (infra) | High |
| Inference cost / latency | Low (short prompts) | High (long prompts) | Retrieval overhead | Low |
| Data needed | 100s–1000s labeled pairs | A few examples | A document corpus | Large unlabeled corpus |
| Keeps data in your VPC | Yes | Depends on API | Yes | Yes |

### When to Choose This Tool

Choose domain-specific fine-tuning when:

- The failures are about **how the model responds** (format, tone, task behavior), not
  missing or changing facts.
- You have a **stable task** with enough quality labeled examples to learn from.
- You need **lower latency/cost** than a giant few-shot prompt, or **on-prem/VPC**
  deployment for privacy or compliance.

Often the best system **combines** them: continued pre-training (if needed) -> fine-tuning
for behavior -> RAG for live facts.

## Resources <a id="resources"></a>

### Official Documentation

- Hugging Face PEFT (LoRA/QLoRA): https://huggingface.co/docs/peft
- TRL `SFTTrainer`: https://huggingface.co/docs/trl/sft_trainer
- bitsandbytes (4-bit quantization): https://huggingface.co/docs/bitsandbytes
- PyTorch `torchtune` fine-tuning library: https://pytorch.org/torchtune

### Tutorials and Guides

- QLoRA paper — *Efficient Finetuning of Quantized LLMs* (Dettmers et al., 2023):
  https://arxiv.org/abs/2305.14314
- LoRA paper — *Low-Rank Adaptation of LLMs* (Hu et al., 2021):
  https://arxiv.org/abs/2106.09685
- Hugging Face *Fine-tuning* guide: https://huggingface.co/docs/transformers/training
- Axolotl (config-driven fine-tuning): https://github.com/axolotl-ai-cloud/axolotl

### Community Resources

- Hugging Face forums: https://discuss.huggingface.co
- r/LocalLLaMA (practical fine-tuning threads): https://www.reddit.com/r/LocalLLaMA
- EleutherAI Discord: https://www.eleuther.ai/community

### Related Technologies

- **Continued / domain-adaptive pre-training** — inject knowledge before fine-tuning.
- **DPO / ORPO** — preference alignment after SFT.
- **RAG** — retrieval for fresh, changing facts.
- **vLLM / TGI** — high-throughput serving with multi-LoRA hot-swap.